In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense



In [2]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)



In [3]:
vocab_size = 2000
tok  = Tokenizer(num_words = vocab_size,oov_token = "")
tok.fit_on_texts(sentences)
seqs = tok.texts_to_sequences(sentences)
maxlen = max(len(s) for s in seqs)
X = pad_sequences(seqs,maxlen = maxlen,padding = 'post')
y = labels


In [4]:
X[0]


array([ 3, 26,  2,  7,  0], dtype=int32)

In [5]:
embed_dim = 16
rnn_units = 8

In [6]:
inp = Input(shape= (maxlen,),dtype="int32",name = 'input')
x = Embedding(input_dim=vocab_size,output_dim=embed_dim,mask_zero=True,name = 'embed')(inp)
rnn = SimpleRNN(units = rnn_units, return_sequences = False,return_state= False,name = 'simple_rnn')
x_last = rnn(x)
out = Dense(1,activation='sigmoid',name = 'out')(x_last)
model = Model(inputs =inp,outputs = out)
model.compile(optimizer='adam',loss = 'binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.fit(X,y,epochs= 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.5000 - loss: 0.6935
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6000 - loss: 0.6732
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6333 - loss: 0.6556
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7000 - loss: 0.6391
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7667 - loss: 0.6222
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9000 - loss: 0.6036
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9000 - loss: 0.5852
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9000 - loss: 0.5651
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9333 - loss: 0.5446 
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9667 - loss: 0.5222
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.4981
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.4737


In [8]:
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [9]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[ 0.04   0.085 -0.024 -0.061  0.005  0.001  0.058  0.011]
 [-0.185 -0.235 -0.034  0.085 -0.347 -0.18   0.028  0.074]
 [-0.319 -0.116  0.077 -0.206 -0.323 -0.071 -0.258  0.134]
 [-0.137 -0.019  0.462 -0.303 -0.337  0.044 -0.407  0.423]
 [-0.137 -0.019  0.462 -0.303 -0.337  0.044 -0.407  0.423]]


In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [11]:
df=pd.read_csv(r'/content/drive/MyDrive/qoute_dataset.csv')

In [12]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [13]:
df.shape

(3038, 2)

In [14]:
quotes = df['quote']
quotes.head()



,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [15]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [16]:
quotes=quotes.str.lower()

In [17]:

import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))



In [18]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [20]:
vocab_size = 8978

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [21]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [22]:
sequence = tokinizer.texts_to_sequences(quotes)



In [23]:
for i in range(3):
  print(quotes[i])



“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [24]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [25]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [26]:
len(X)

85270

In [27]:
len(y)

85270

In [28]:
max_len = max(len(x) for x in X)
print(max_len)


745


In [29]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')


In [30]:
y = np.array(y)



In [31]:
X_padded.shape

(85270, 745)

In [32]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)


In [33]:
y.shape


(85270,)

In [34]:

y_one_hot.shape

(85270, 8978)

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [36]:

embedding_dim = 50
rnn_units = 128

In [37]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [38]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [41]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)



In [42]:
lstm_model.summary()



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [43]:
from tensorflow.keras.models import load_model

lstm_model = load_model(r'/content/drive/MyDrive/Aaditya_0103AD241001_A/lstm_model (1).h5')

In [44]:
lstm_model.save("lstm_model.h5")

In [45]:

index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word


In [46]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [47]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [48]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

worrying


In [49]:

def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [50]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a  thousand times i wrote the less if it does not


In [51]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [52]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)
